In [1]:
# ============================================================
# 05_FORECASTING
# FOOD PRICE FORECASTING MODEL DEVELOPMENT
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "food_price_forecasting_features.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "forecasting"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. LOAD FEATURE-ENGINEERED DATA
# ============================================================

print("=" * 70)
print("FOOD PRICE FORECASTING")
print("=" * 70)

print("\nLoading feature-engineered dataset...")

df = pd.read_csv(FEATURE_FILE)

print(f"Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")


# ============================================================
# 3. BASIC VALIDATION
# ============================================================

required_columns = [
    "Crop",
    "Month_Start",
    "Target_Next_Month_Price",
    "Median_Price",
    "Price_Lag_1M"
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing from the feature dataset: "
        f"{missing_required}"
    )

df["Month_Start"] = pd.to_datetime(
    df["Month_Start"],
    errors="coerce"
)

df["Target_Next_Month_Price"] = pd.to_numeric(
    df["Target_Next_Month_Price"],
    errors="coerce"
)

df["Median_Price"] = pd.to_numeric(
    df["Median_Price"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "Crop",
        "Month_Start",
        "Target_Next_Month_Price"
    ]
).copy()

df = df.sort_values(
    ["Crop", "Month_Start"]
).reset_index(drop=True)

if df["Target_Next_Month_Price"].isna().any():
    raise ValueError("Target_Next_Month_Price contains missing values after validation.")

if not np.isfinite(df["Target_Next_Month_Price"]).all():
    raise ValueError("Target_Next_Month_Price contains infinite values.")

print("✓ Target values are complete and finite.")


# ============================================================
# 4. REMOVE NON-FEATURE / TARGET COLUMNS
# ============================================================

# These columns must NOT be used as predictors.
# Month_Start is retained separately for chronological ordering.

excluded_columns = [
    "Month_Start",
    "Target_Next_Month_Price",
    "Median_Price"
]

feature_columns = [
    col
    for col in df.columns
    if col not in excluded_columns
]

# Do not allow split helper columns to become model features.
feature_columns = [
    col
    for col in feature_columns
    if not col.startswith("_")
]

print("\nInitial model features:", len(feature_columns))

# ------------------------------------------------------------
# TEMPORAL LEAKAGE SAFETY CHECK
# ------------------------------------------------------------
# Features must not contain the target itself or future-date helpers.
for forbidden in [
    "Target_Next_Month_Price",
    "Target_Price",
    "Future_Price",
    "Future_Month",
]:
    if forbidden in feature_columns:
        raise ValueError(
            f"Potential target/future leakage detected in feature list: {forbidden}"
        )

# Month_Start is the only chronological reference and is excluded
# from the predictor matrix above.
print("✓ Target/future leakage column check passed.")


# ============================================================
# 5. CHRONOLOGICAL TRAIN / TEST SPLIT BY CROP
# ============================================================

# Each crop is split independently.
# Final 20% of each crop's chronological observations = TEST.
# First 80% = TRAIN.

df["_Split"] = "train"

for crop in df["Crop"].dropna().unique():

    crop_idx = df.index[
        df["Crop"] == crop
    ].tolist()

    if len(crop_idx) < 5:
        continue

    test_size = max(
        1,
        int(np.ceil(len(crop_idx) * 0.20))
    )

    test_idx = crop_idx[-test_size:]

    df.loc[test_idx, "_Split"] = "test"


train_df = df[
    df["_Split"] == "train"
].copy()

test_df = df[
    df["_Split"] == "test"
].copy()


print("\n" + "=" * 70)
print("CHRONOLOGICAL TRAIN / TEST SPLIT")
print("=" * 70)

print(f"\nTraining rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")

print("\nSplit by crop:")

split_summary = (
    df.groupby(
        ["Crop", "_Split"]
    )
    .size()
    .unstack(fill_value=0)
)

print(split_summary)

print("\nChronological boundaries by crop:")
for crop in sorted(df["Crop"].dropna().unique()):
    crop_train = train_df[train_df["Crop"] == crop]
    crop_test = test_df[test_df["Crop"] == crop]
    if len(crop_train) and len(crop_test):
        print(
            f"  {crop}: "
            f"train {crop_train['Month_Start'].min().date()} → {crop_train['Month_Start'].max().date()} | "
            f"test {crop_test['Month_Start'].min().date()} → {crop_test['Month_Start'].max().date()} "
            f"({len(crop_train)} / {len(crop_test)})"
        )


# ============================================================
# 6. VALIDATE CHRONOLOGICAL ORDER
# ============================================================

print("\nChronological validation:")

invalid_splits = []

for crop in df["Crop"].dropna().unique():

    crop_train = train_df[
        train_df["Crop"] == crop
    ]

    crop_test = test_df[
        test_df["Crop"] == crop
    ]

    if len(crop_train) == 0 or len(crop_test) == 0:
        continue

    train_last_date = crop_train["Month_Start"].max()
    test_first_date = crop_test["Month_Start"].min()

    if train_last_date >= test_first_date:
        invalid_splits.append(crop)

if invalid_splits:
    raise ValueError(
        f"Chronological split validation failed for: "
        f"{invalid_splits}"
    )

print("✓ All crop-level splits are chronological.")


# ============================================================
# 7. PREPARE X / y
# ============================================================

X_train = train_df[
    feature_columns
].copy()

y_train = train_df[
    "Target_Next_Month_Price"
].copy()

X_test = test_df[
    feature_columns
].copy()

y_test = test_df[
    "Target_Next_Month_Price"
].copy()


# ============================================================
# 8. AUTOMATICALLY IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

numeric_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()


print("\n" + "=" * 70)
print("FEATURE TYPES")
print("=" * 70)

print(f"\nTotal features:      {len(feature_columns)}")
print(f"Numeric features:    {len(numeric_features)}")
print(f"Categorical features:{len(categorical_features)}")

if categorical_features:
    print("\nCategorical features:")
    for col in categorical_features:
        print(f"  - {col}")

if numeric_features:
    print("\nNumeric features:")
    for col in numeric_features:
        print(f"  - {col}")


# ============================================================
# 9. SAFETY CHECK
# ============================================================

if len(numeric_features) == 0 and len(categorical_features) == 0:
    raise ValueError(
        "No usable model features were detected."
    )

# Remove columns that are completely empty in training data.
all_missing_features = [
    col
    for col in feature_columns
    if X_train[col].isna().all()
]

if all_missing_features:

    print("\nRemoving completely missing features:")

    for col in all_missing_features:
        print(f"  - {col}")

    feature_columns = [
        col
        for col in feature_columns
        if col not in all_missing_features
    ]

    X_train = X_train[feature_columns]
    X_test = X_test[feature_columns]

    numeric_features = X_train.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "string", "bool"]
    ).columns.tolist()


# ============================================================
# 10. PREPROCESSING PIPELINES
# ============================================================

# Numeric:
# Missing values -> median

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


# Categorical:
# Missing values -> most frequent
# Categories -> one-hot encoding

try:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:

    # Compatibility with older scikit-learn versions.
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            categorical_encoder
        )
    ]
)


# Combine numeric + categorical preprocessing.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


# ============================================================
# 11. MODEL DEFINITIONS
# ============================================================

models = {

    "Naive_Last_Month": None,

    "Random_Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "Extra_Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "Hist_Gradient_Boosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}


# ============================================================
# 12. METRIC FUNCTIONS
# ============================================================

def calculate_mape(
    y_true,
    y_pred
):
    """
    Mean Absolute Percentage Error.
    Zero actual values are excluded.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        )
        * 100
    )


def calculate_smape(
    y_true,
    y_pred
):
    """
    Symmetric Mean Absolute Percentage Error.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    )

    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            2
            * np.abs(
                y_pred[mask] - y_true[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def evaluate_predictions(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = calculate_mape(
        y_true,
        y_pred
    )

    smape = calculate_smape(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "SMAPE": smape,
        "R2": r2
    }


# ============================================================
# 13. BASELINE MODEL
# ============================================================

print("\n" + "=" * 70)
print("NAIVE BASELINE")
print("=" * 70)

if "Price_Lag_1M" not in test_df.columns:

    raise ValueError(
        "Price_Lag_1M is required for the Naive Last Month baseline."
    )

naive_predictions = test_df[
    "Price_Lag_1M"
].copy()

# Fill any unexpected missing lag values using
# the training target median.

naive_predictions = naive_predictions.fillna(
    y_train.median()
)

naive_metrics = evaluate_predictions(
    y_test,
    naive_predictions
)

print(
    f"\nNaive Last Month RMSE: "
    f"₹{naive_metrics['RMSE']:,.2f}"
)

print(
    f"Naive Last Month MAE:  "
    f"₹{naive_metrics['MAE']:,.2f}"
)

print(
    f"Naive Last Month MAPE: "
    f"{naive_metrics['MAPE']:.2f}%"
)

print(
    f"Naive Last Month R²:   "
    f"{naive_metrics['R2']:.4f}"
)


# ============================================================
# 14. STORE MODEL RESULTS
# ============================================================

model_results = []

model_results.append(
    {
        "Model": "Naive_Last_Month",
        "MAE": naive_metrics["MAE"],
        "RMSE": naive_metrics["RMSE"],
        "MAPE": naive_metrics["MAPE"],
        "SMAPE": naive_metrics["SMAPE"],
        "R2": naive_metrics["R2"]
    }
)


# ============================================================
# 15. TRAIN MACHINE LEARNING MODELS
# ============================================================

trained_models = {}

prediction_storage = []

print("\n" + "=" * 70)
print("MODEL TRAINING")
print("=" * 70)


for model_name, model in models.items():

    if model_name == "Naive_Last_Month":
        continue

    print(
        f"\nTraining {model_name}..."
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    predictions = pipeline.predict(
        X_test
    )

    # Prevent impossible negative prices.
    predictions = np.maximum(
        predictions,
        0
    )

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    metrics = evaluate_predictions(
        y_test,
        predictions
    )

    model_results.append(
        {
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
            "SMAPE": metrics["SMAPE"],
            "R2": metrics["R2"]
        }
    )

    trained_models[
        model_name
    ] = pipeline

    prediction_storage.append(
        pd.DataFrame(
            {
                "Model": model_name,
                "Crop": test_df["Crop"].values,
                "Month_Start": test_df["Month_Start"].values,
                "Actual_Price": y_test.values,
                "Predicted_Price": predictions
            }
        )
    )

    print(
        f"  MAE:   ₹{metrics['MAE']:,.2f}"
    )

    print(
        f"  RMSE:  ₹{metrics['RMSE']:,.2f}"
    )

    print(
        f"  MAPE:  {metrics['MAPE']:.2f}%"
    )

    print(
        f"  SMAPE: {metrics['SMAPE']:.2f}%"
    )

    print(
        f"  R²:    {metrics['R2']:.4f}"
    )


# ============================================================
# 16. MODEL COMPARISON
# ============================================================

model_comparison = pd.DataFrame(
    model_results
)

model_comparison = model_comparison.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

model_comparison.to_csv(
    OUTPUT_DIR
    / "model_comparison.csv",
    index=False
)


print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(
    model_comparison.round(4)
)


# ============================================================
# 17. SELECT BEST MODEL
# ============================================================

# Best model is selected using chronological TEST RMSE.
# Lower RMSE = smaller typical squared prediction error.

if model_comparison.empty:
    raise ValueError("No model evaluation results were produced.")

if not np.isfinite(
    model_comparison[["MAE", "RMSE", "MAPE", "SMAPE", "R2"]].to_numpy(dtype=float)
).all():
    raise ValueError("Model evaluation produced non-finite metrics.")

ml_results = model_comparison[
    model_comparison["Model"] != "Naive_Last_Month"
].copy()

best_model_name = (
    ml_results
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

best_model = trained_models[
    best_model_name
]

best_row = model_comparison[
    model_comparison["Model"] == best_model_name
].iloc[0]


print("\n" + "=" * 70)
print("BEST MACHINE LEARNING MODEL")
print("=" * 70)

print(
    f"\nSelected model: {best_model_name}"
)

print(
    f"Test MAE:   ₹{best_row['MAE']:,.2f}"
)

print(
    f"Test RMSE:  ₹{best_row['RMSE']:,.2f}"
)

print(
    f"Test MAPE:  {best_row['MAPE']:.2f}%"
)

print(
    f"Test SMAPE: {best_row['SMAPE']:.2f}%"
)

print(
    f"Test R²:    {best_row['R2']:.4f}"
)


# ============================================================
# 18. BEST MODEL PREDICTIONS
# ============================================================

best_predictions = best_model.predict(
    X_test
)

best_predictions = np.maximum(
    best_predictions,
    0
)

test_predictions = test_df[
    [
        "Crop",
        "Month_Start"
    ]
].copy()

test_predictions[
    "Actual_Price"
] = y_test.values

test_predictions[
    "Predicted_Price"
] = best_predictions

test_predictions[
    "Absolute_Error"
] = np.abs(
    test_predictions["Actual_Price"]
    -
    test_predictions["Predicted_Price"]
)

test_predictions[
    "Percentage_Error"
] = np.where(
    test_predictions["Actual_Price"] != 0,
    (
        test_predictions["Absolute_Error"]
        /
        test_predictions["Actual_Price"]
    )
    * 100,
    np.nan
)

test_predictions[
    "Model"
] = best_model_name


# ============================================================
# 19. SAVE TEST PREDICTIONS
# ============================================================

test_predictions.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ============================================================
# 20. CROP-LEVEL MODEL PERFORMANCE
# ============================================================

crop_results = []

for crop in sorted(
    test_predictions["Crop"].dropna().unique()
):

    crop_data = test_predictions[
        test_predictions["Crop"] == crop
    ].copy()

    crop_metrics = evaluate_predictions(
        crop_data["Actual_Price"],
        crop_data["Predicted_Price"]
    )

    crop_results.append(
        {
            "Crop": crop,
            "Test_Observations": len(crop_data),
            "MAE": crop_metrics["MAE"],
            "RMSE": crop_metrics["RMSE"],
            "MAPE": crop_metrics["MAPE"],
            "SMAPE": crop_metrics["SMAPE"],
            "R2": crop_metrics["R2"]
        }
    )


crop_model_performance = pd.DataFrame(
    crop_results
).sort_values(
    "RMSE"
)

crop_model_performance.to_csv(
    OUTPUT_DIR
    / "crop_model_performance.csv",
    index=False
)


print("\n" + "=" * 70)
print("CROP-LEVEL PERFORMANCE")
print("=" * 70)

display(
    crop_model_performance.round(4)
)


# ============================================================
# 21. PRICE-BAND PERFORMANCE
# ============================================================

# Price bands help determine whether the model performs
# similarly across low-, medium-, and high-price observations.

test_predictions[
    "Price_Band"
] = pd.cut(
    test_predictions["Actual_Price"],
    bins=[
        -np.inf,
        2000,
        5000,
        7500,
        10000,
        20000,
        50000,
        100000,
        np.inf
    ],
    labels=[
        "< ₹2,000",
        "₹2,000–₹5,000",
        "₹5,000–₹7,500",
        "₹7,500–₹10,000",
        "₹10,000–₹20,000",
        "₹20,000–₹50,000",
        "₹50,000–₹100,000",
        "≥ ₹100,000"
    ]
)


band_results = []

for band, group in test_predictions.groupby(
    "Price_Band",
    observed=True
):

    if len(group) == 0:
        continue

    band_metrics = evaluate_predictions(
        group["Actual_Price"],
        group["Predicted_Price"]
    )

    band_results.append(
        {
            "Price_Band": str(band),
            "Test_Observations": len(group),
            "MAE": band_metrics["MAE"],
            "RMSE": band_metrics["RMSE"],
            "MAPE": band_metrics["MAPE"],
            "SMAPE": band_metrics["SMAPE"],
            "R2": band_metrics["R2"]
        }
    )


price_band_performance = pd.DataFrame(
    band_results
)

price_band_performance.to_csv(
    OUTPUT_DIR
    / "price_band_model_performance.csv",
    index=False
)


print("\n" + "=" * 70)
print("PRICE-BAND PERFORMANCE")
print("=" * 70)

display(
    price_band_performance.round(4)
)


# ============================================================
# 22. LARGEST PREDICTION ERRORS
# ============================================================

largest_errors = (
    test_predictions
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
    .head(20)
    [
        [
            "Crop",
            "Month_Start",
            "Actual_Price",
            "Predicted_Price",
            "Absolute_Error",
            "Percentage_Error"
        ]
    ]
)


print("\n" + "=" * 70)
print("20 LARGEST PREDICTION ERRORS")
print("=" * 70)

display(
    largest_errors.round(2)
)


# ============================================================
# 23. SAVE TRAIN / TEST DATA
# ============================================================

train_df.to_csv(
    OUTPUT_DIR
    / "forecasting_train_data.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIR
    / "forecasting_test_data.csv",
    index=False
)


# ============================================================
# 24. SAVE BEST MODEL
# ============================================================

best_model_path = (
    MODEL_DIR
    / "food_price_forecasting_best_model.joblib"
)

joblib.dump(
    best_model,
    best_model_path
)


# ============================================================
# 25. SAVE FEATURE INFORMATION
# ============================================================

feature_info = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Type": [
            (
                "Categorical"
                if col in categorical_features
                else "Numeric"
            )
            for col in feature_columns
        ]
    }
)

feature_info.to_csv(
    OUTPUT_DIR
    / "model_feature_information.csv",
    index=False
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FORECASTING PIPELINE COMPLETE")
print("=" * 70)

print(
    f"\nTotal dataset rows:       {len(df):,}"
)

print(
    f"Training rows:            {len(train_df):,}"
)

print(
    f"Testing rows:             {len(test_df):,}"
)

print(
    f"Number of crops:          {df['Crop'].nunique()}"
)

print(
    f"Number of features:       {len(feature_columns)}"
)

print(
    f"Numeric features:         {len(numeric_features)}"
)

print(
    f"Categorical features:     {len(categorical_features)}"
)

print(
    f"\nBest ML model:            {best_model_name}"
)

print(
    f"Best model RMSE:          ₹{best_row['RMSE']:,.2f}"
)

print(
    f"Best model MAE:           ₹{best_row['MAE']:,.2f}"
)

print(
    f"Best model MAPE:          {best_row['MAPE']:.2f}%"
)

print(
    f"Best model SMAPE:         {best_row['SMAPE']:.2f}%"
)

print(
    f"Best model R²:            {best_row['R2']:.4f}"
)

print("\nSaved files:")

print(
    f"  ✓ {OUTPUT_DIR / 'model_comparison.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'crop_model_performance.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'price_band_model_performance.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'test_predictions.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'forecasting_train_data.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'forecasting_test_data.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'model_feature_information.csv'}"
)

print(
    f"  ✓ {best_model_path}"
)

print(
    f"Test observations evaluated: {len(test_predictions):,}"
)

print(
    f"Models evaluated:            {len(model_comparison):,}"
)

print("\n" + "=" * 70)
print("READY FOR MODEL EVALUATION")
print("=" * 70)

FOOD PRICE FORECASTING

Loading feature-engineered dataset...
Dataset loaded successfully.
Rows: 1,179
Columns: 66
✓ Target values are complete and finite.

Initial model features: 63
✓ Target/future leakage column check passed.

CHRONOLOGICAL TRAIN / TEST SPLIT

Training rows: 936
Testing rows:  243

Split by crop:
_Split         test  train
Crop                      
Banana           27    104
Cotton           27    104
Groundnut        27    104
Jowar/Sorghum    27    104
Maize            27    104
Onion            27    104
Potato           27    104
Rice             27    104
Wheat            27    104

Chronological boundaries by crop:
  Banana: train 2015-01-01 → 2023-08-01 | test 2023-09-01 → 2025-11-01 (104 / 27)
  Cotton: train 2015-01-01 → 2023-08-01 | test 2023-09-01 → 2025-11-01 (104 / 27)
  Groundnut: train 2015-01-01 → 2023-08-01 | test 2023-09-01 → 2025-11-01 (104 / 27)
  Jowar/Sorghum: train 2015-01-01 → 2023-08-01 | test 2023-09-01 → 2025-11-01 (104 / 27)
  Maize: tra

,Model,MAE,RMSE,MAPE,SMAPE,R2
0,Extra_Trees,78.7358,134.4564,1.9872,1.9699,0.9941
1,Hist_Gradient_Boosting,99.8696,211.6214,1.9594,1.9227,0.9855
2,Random_Forest,231.7966,346.2952,7.2151,7.0824,0.9611
3,Naive_Last_Month,271.2243,458.6914,10.0720,9.5730,0.9317



BEST MACHINE LEARNING MODEL

Selected model: Extra_Trees
Test MAE:   ₹78.74
Test RMSE:  ₹134.46
Test MAPE:  1.99%
Test SMAPE: 1.97%
Test R²:    0.9941

CROP-LEVEL PERFORMANCE


,Crop,Test_Observations,MAE,RMSE,MAPE,SMAPE,R2
4,Maize,27,3.4725,5.1849,0.1738,0.1735,0.9990
8,Wheat,27,10.1475,12.7435,0.4002,0.3991,0.9929
7,Rice,27,19.6552,24.2583,0.5908,0.5889,0.9797
6,Potato,27,23.2173,33.1057,1.4910,1.4723,0.9964
3,Jowar/Sorghum,27,81.5651,100.0374,2.9540,2.9021,0.9597
1,Cotton,27,106.0085,139.3837,1.5082,1.5045,0.5931
5,Onion,27,91.4874,143.1790,3.2720,3.2652,0.9808
0,Banana,27,110.3860,147.8647,2.9345,2.9905,0.9279
2,Groundnut,27,262.6826,298.3849,4.5605,4.4328,0.4237



PRICE-BAND PERFORMANCE


,Price_Band,Test_Observations,MAE,RMSE,MAPE,SMAPE,R2
0,"< ₹2,000",39,28.8868,52.9000,1.8598,1.8178,0.9704
1,"₹2,000–₹5,000",151,54.8412,96.5683,1.6629,1.6686,0.9769
2,"₹5,000–₹7,500",52,180.2854,230.0942,2.9749,2.9065,0.8900
3,"₹7,500–₹10,000",1,350.3604,350.3604,4.5721,4.6791,NaN



20 LARGEST PREDICTION ERRORS


,Crop,Month_Start,Actual_Price,Predicted_Price,Absolute_Error,Percentage_Error
372,Groundnut,2024-03-01,5855.00,6366.36,511.36,8.73
760,Onion,2023-10-01,4100.00,3594.76,505.24,12.32
370,Groundnut,2024-01-01,6062.00,6551.13,489.13,8.07
367,Groundnut,2023-10-01,6215.25,6672.93,457.68,7.36
371,Groundnut,2024-02-01,6107.50,6553.87,446.37,7.31
379,Groundnut,2024-10-01,5380.00,5810.78,430.78,8.01
374,Groundnut,2024-05-01,5750.00,6150.45,400.45,6.96
378,Groundnut,2024-09-01,5400.00,5795.58,395.58,7.33
375,Groundnut,2024-06-01,5745.00,6135.15,390.15,6.79
368,Groundnut,2023-11-01,6405.00,6781.61,376.61,5.88



FORECASTING PIPELINE COMPLETE

Total dataset rows:       1,179
Training rows:            936
Testing rows:             243
Number of crops:          9
Number of features:       63
Numeric features:         60
Categorical features:     3

Best ML model:            Extra_Trees
Best model RMSE:          ₹134.46
Best model MAE:           ₹78.74
Best model MAPE:          1.99%
Best model SMAPE:         1.97%
Best model R²:            0.9941

Saved files:
  ✓ d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs\forecasting\model_comparison.csv
  ✓ d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs\forecasting\crop_model_performance.csv
  ✓ d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs\forecasting\price_band_model_performance.csv
  ✓ d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\outputs\forecasting\test_predictions.csv
  ✓ d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecastin

In [ ]:
# ============================================================
# 05_FORECASTING
# FOOD PRICE FORECASTING MODEL DEVELOPMENT
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "food_price_forecasting_features.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "forecasting"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. LOAD FEATURE-ENGINEERED DATA
# ============================================================

print("=" * 70)
print("FOOD PRICE FORECASTING")
print("=" * 70)

print("\nLoading feature-engineered dataset...")

df = pd.read_csv(FEATURE_FILE)

print(f"Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")


# ============================================================
# 3. BASIC VALIDATION
# ============================================================

required_columns = [
    "Crop",
    "Month_Start",
    "Target_Next_Month_Price",
    "Median_Price",
    "Price_Lag_1M"
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing from the feature dataset: "
        f"{missing_required}"
    )

df["Month_Start"] = pd.to_datetime(
    df["Month_Start"],
    errors="coerce"
)

df["Target_Next_Month_Price"] = pd.to_numeric(
    df["Target_Next_Month_Price"],
    errors="coerce"
)

df["Median_Price"] = pd.to_numeric(
    df["Median_Price"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "Crop",
        "Month_Start",
        "Target_Next_Month_Price"
    ]
).copy()

df = df.sort_values(
    ["Crop", "Month_Start"]
).reset_index(drop=True)


# ============================================================
# 4. REMOVE NON-FEATURE / TARGET COLUMNS
# ============================================================

# These columns must NOT be used as predictors.
# Month_Start is retained separately for chronological ordering.

excluded_columns = [
    "Month_Start",
    "Target_Next_Month_Price",
    "Median_Price"
]

feature_columns = [
    col
    for col in df.columns
    if col not in excluded_columns
]

# Do not allow split helper columns to become model features.
feature_columns = [
    col
    for col in feature_columns
    if not col.startswith("_")
]

print("\nInitial model features:", len(feature_columns))


# ============================================================
# 5. CHRONOLOGICAL TRAIN / TEST SPLIT BY CROP
# ============================================================

# Each crop is split independently.
# Final 20% of each crop's chronological observations = TEST.
# First 80% = TRAIN.

df["_Split"] = "train"

for crop in df["Crop"].dropna().unique():

    crop_idx = df.index[
        df["Crop"] == crop
    ].tolist()

    if len(crop_idx) < 5:
        continue

    test_size = max(
        1,
        int(np.ceil(len(crop_idx) * 0.20))
    )

    test_idx = crop_idx[-test_size:]

    df.loc[test_idx, "_Split"] = "test"


train_df = df[
    df["_Split"] == "train"
].copy()

test_df = df[
    df["_Split"] == "test"
].copy()


print("\n" + "=" * 70)
print("CHRONOLOGICAL TRAIN / TEST SPLIT")
print("=" * 70)

print(f"\nTraining rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")

print("\nSplit by crop:")

split_summary = (
    df.groupby(
        ["Crop", "_Split"]
    )
    .size()
    .unstack(fill_value=0)
)

print(split_summary)


# ============================================================
# 6. VALIDATE CHRONOLOGICAL ORDER
# ============================================================

print("\nChronological validation:")

invalid_splits = []

for crop in df["Crop"].dropna().unique():

    crop_train = train_df[
        train_df["Crop"] == crop
    ]

    crop_test = test_df[
        test_df["Crop"] == crop
    ]

    if len(crop_train) == 0 or len(crop_test) == 0:
        continue

    train_last_date = crop_train["Month_Start"].max()
    test_first_date = crop_test["Month_Start"].min()

    if train_last_date >= test_first_date:
        invalid_splits.append(crop)

if invalid_splits:
    raise ValueError(
        f"Chronological split validation failed for: "
        f"{invalid_splits}"
    )

print("✓ All crop-level splits are chronological.")


# ============================================================
# 7. PREPARE X / y
# ============================================================

X_train = train_df[
    feature_columns
].copy()

y_train = train_df[
    "Target_Next_Month_Price"
].copy()

X_test = test_df[
    feature_columns
].copy()

y_test = test_df[
    "Target_Next_Month_Price"
].copy()


# ============================================================
# 8. AUTOMATICALLY IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

numeric_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()


print("\n" + "=" * 70)
print("FEATURE TYPES")
print("=" * 70)

print(f"\nTotal features:      {len(feature_columns)}")
print(f"Numeric features:    {len(numeric_features)}")
print(f"Categorical features:{len(categorical_features)}")

if categorical_features:
    print("\nCategorical features:")
    for col in categorical_features:
        print(f"  - {col}")

if numeric_features:
    print("\nNumeric features:")
    for col in numeric_features:
        print(f"  - {col}")


# ============================================================
# 9. SAFETY CHECK
# ============================================================

if len(numeric_features) == 0 and len(categorical_features) == 0:
    raise ValueError(
        "No usable model features were detected."
    )

# Remove columns that are completely empty in training data.
all_missing_features = [
    col
    for col in feature_columns
    if X_train[col].isna().all()
]

if all_missing_features:

    print("\nRemoving completely missing features:")

    for col in all_missing_features:
        print(f"  - {col}")

    feature_columns = [
        col
        for col in feature_columns
        if col not in all_missing_features
    ]

    X_train = X_train[feature_columns]
    X_test = X_test[feature_columns]

    numeric_features = X_train.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "string", "bool"]
    ).columns.tolist()


# ============================================================
# 10. PREPROCESSING PIPELINES
# ============================================================

# Numeric:
# Missing values -> median

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


# Categorical:
# Missing values -> most frequent
# Categories -> one-hot encoding

try:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:

    # Compatibility with older scikit-learn versions.
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            categorical_encoder
        )
    ]
)


# Combine numeric + categorical preprocessing.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


# ============================================================
# 11. MODEL DEFINITIONS
# ============================================================

models = {

    "Naive_Last_Month": None,

    "Random_Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "Extra_Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "Hist_Gradient_Boosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}


# ============================================================
# 12. METRIC FUNCTIONS
# ============================================================

def calculate_mape(
    y_true,
    y_pred
):
    """
    Mean Absolute Percentage Error.
    Zero actual values are excluded.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        )
        * 100
    )


def calculate_smape(
    y_true,
    y_pred
):
    """
    Symmetric Mean Absolute Percentage Error.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    )

    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            2
            * np.abs(
                y_pred[mask] - y_true[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def evaluate_predictions(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = calculate_mape(
        y_true,
        y_pred
    )

    smape = calculate_smape(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "SMAPE": smape,
        "R2": r2
    }


# ============================================================
# 13. BASELINE MODEL
# ============================================================

print("\n" + "=" * 70)
print("NAIVE BASELINE")
print("=" * 70)

if "Price_Lag_1M" not in test_df.columns:

    raise ValueError(
        "Price_Lag_1M is required for the Naive Last Month baseline."
    )

naive_predictions = test_df[
    "Price_Lag_1M"
].copy()

# Fill any unexpected missing lag values using
# the training target median.

naive_predictions = naive_predictions.fillna(
    y_train.median()
)

naive_metrics = evaluate_predictions(
    y_test,
    naive_predictions
)

print(
    f"\nNaive Last Month RMSE: "
    f"₹{naive_metrics['RMSE']:,.2f}"
)

print(
    f"Naive Last Month MAE:  "
    f"₹{naive_metrics['MAE']:,.2f}"
)

print(
    f"Naive Last Month MAPE: "
    f"{naive_metrics['MAPE']:.2f}%"
)

print(
    f"Naive Last Month R²:   "
    f"{naive_metrics['R2']:.4f}"
)


# ============================================================
# 14. STORE MODEL RESULTS
# ============================================================

model_results = []

model_results.append(
    {
        "Model": "Naive_Last_Month",
        "MAE": naive_metrics["MAE"],
        "RMSE": naive_metrics["RMSE"],
        "MAPE": naive_metrics["MAPE"],
        "SMAPE": naive_metrics["SMAPE"],
        "R2": naive_metrics["R2"]
    }
)


# ============================================================
# 15. TRAIN MACHINE LEARNING MODELS
# ============================================================

trained_models = {}

prediction_storage = []

print("\n" + "=" * 70)
print("MODEL TRAINING")
print("=" * 70)


for model_name, model in models.items():

    if model_name == "Naive_Last_Month":
        continue

    print(
        f"\nTraining {model_name}..."
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    predictions = pipeline.predict(
        X_test
    )

    # Prevent impossible negative prices.
    predictions = np.maximum(
        predictions,
        0
    )

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    metrics = evaluate_predictions(
        y_test,
        predictions
    )

    model_results.append(
        {
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
            "SMAPE": metrics["SMAPE"],
            "R2": metrics["R2"]
        }
    )

    trained_models[
        model_name
    ] = pipeline

    prediction_storage.append(
        pd.DataFrame(
            {
                "Model": model_name,
                "Crop": test_df["Crop"].values,
                "Month_Start": test_df["Month_Start"].values,
                "Actual_Price": y_test.values,
                "Predicted_Price": predictions
            }
        )
    )

    print(
        f"  MAE:   ₹{metrics['MAE']:,.2f}"
    )

    print(
        f"  RMSE:  ₹{metrics['RMSE']:,.2f}"
    )

    print(
        f"  MAPE:  {metrics['MAPE']:.2f}%"
    )

    print(
        f"  SMAPE: {metrics['SMAPE']:.2f}%"
    )

    print(
        f"  R²:    {metrics['R2']:.4f}"
    )


# ============================================================
# 16. MODEL COMPARISON
# ============================================================

model_comparison = pd.DataFrame(
    model_results
)

model_comparison = model_comparison.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

model_comparison.to_csv(
    OUTPUT_DIR
    / "model_comparison.csv",
    index=False
)


print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(
    model_comparison.round(4)
)


# ============================================================
# 17. SELECT BEST MODEL
# ============================================================

# Best model is selected using chronological TEST RMSE.
# Lower RMSE = smaller typical squared prediction error.

ml_results = model_comparison[
    model_comparison["Model"] != "Naive_Last_Month"
].copy()

best_model_name = (
    ml_results
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

best_model = trained_models[
    best_model_name
]

best_row = model_comparison[
    model_comparison["Model"] == best_model_name
].iloc[0]


print("\n" + "=" * 70)
print("BEST MACHINE LEARNING MODEL")
print("=" * 70)

print(
    f"\nSelected model: {best_model_name}"
)

print(
    f"Test MAE:   ₹{best_row['MAE']:,.2f}"
)

print(
    f"Test RMSE:  ₹{best_row['RMSE']:,.2f}"
)

print(
    f"Test MAPE:  {best_row['MAPE']:.2f}%"
)

print(
    f"Test SMAPE: {best_row['SMAPE']:.2f}%"
)

print(
    f"Test R²:    {best_row['R2']:.4f}"
)


# ============================================================
# 18. BEST MODEL PREDICTIONS
# ============================================================

best_predictions = best_model.predict(
    X_test
)

best_predictions = np.maximum(
    best_predictions,
    0
)

test_predictions = test_df[
    [
        "Crop",
        "Month_Start"
    ]
].copy()

test_predictions[
    "Actual_Price"
] = y_test.values

test_predictions[
    "Predicted_Price"
] = best_predictions

test_predictions[
    "Absolute_Error"
] = np.abs(
    test_predictions["Actual_Price"]
    -
    test_predictions["Predicted_Price"]
)

test_predictions[
    "Percentage_Error"
] = np.where(
    test_predictions["Actual_Price"] != 0,
    (
        test_predictions["Absolute_Error"]
        /
        test_predictions["Actual_Price"]
    )
    * 100,
    np.nan
)

test_predictions[
    "Model"
] = best_model_name


# ============================================================
# 19. SAVE TEST PREDICTIONS
# ============================================================

test_predictions.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ============================================================
# 20. CROP-LEVEL MODEL PERFORMANCE
# ============================================================

crop_results = []

for crop in sorted(
    test_predictions["Crop"].dropna().unique()
):

    crop_data = test_predictions[
        test_predictions["Crop"] == crop
    ].copy()

    crop_metrics = evaluate_predictions(
        crop_data["Actual_Price"],
        crop_data["Predicted_Price"]
    )

    crop_results.append(
        {
            "Crop": crop,
            "Test_Observations": len(crop_data),
            "MAE": crop_metrics["MAE"],
            "RMSE": crop_metrics["RMSE"],
            "MAPE": crop_metrics["MAPE"],
            "SMAPE": crop_metrics["SMAPE"],
            "R2": crop_metrics["R2"]
        }
    )


crop_model_performance = pd.DataFrame(
    crop_results
).sort_values(
    "RMSE"
)

crop_model_performance.to_csv(
    OUTPUT_DIR
    / "crop_model_performance.csv",
    index=False
)


print("\n" + "=" * 70)
print("CROP-LEVEL PERFORMANCE")
print("=" * 70)

display(
    crop_model_performance.round(4)
)


# ============================================================
# 21. PRICE-BAND PERFORMANCE
# ============================================================

# Price bands help determine whether the model performs
# similarly across low-, medium-, and high-price observations.

test_predictions[
    "Price_Band"
] = pd.cut(
    test_predictions["Actual_Price"],
    bins=[
        -np.inf,
        2000,
        5000,
        7500,
        10000,
        20000,
        50000,
        100000,
        np.inf
    ],
    labels=[
        "< ₹2,000",
        "₹2,000–₹5,000",
        "₹5,000–₹7,500",
        "₹7,500–₹10,000",
        "₹10,000–₹20,000",
        "₹20,000–₹50,000",
        "₹50,000–₹100,000",
        "≥ ₹100,000"
    ]
)


band_results = []

for band, group in test_predictions.groupby(
    "Price_Band",
    observed=True
):

    if len(group) == 0:
        continue

    band_metrics = evaluate_predictions(
        group["Actual_Price"],
        group["Predicted_Price"]
    )

    band_results.append(
        {
            "Price_Band": str(band),
            "Test_Observations": len(group),
            "MAE": band_metrics["MAE"],
            "RMSE": band_metrics["RMSE"],
            "MAPE": band_metrics["MAPE"],
            "SMAPE": band_metrics["SMAPE"],
            "R2": band_metrics["R2"]
        }
    )


price_band_performance = pd.DataFrame(
    band_results
)

price_band_performance.to_csv(
    OUTPUT_DIR
    / "price_band_model_performance.csv",
    index=False
)


print("\n" + "=" * 70)
print("PRICE-BAND PERFORMANCE")
print("=" * 70)

display(
    price_band_performance.round(4)
)


# ============================================================
# 22. LARGEST PREDICTION ERRORS
# ============================================================

largest_errors = (
    test_predictions
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
    .head(20)
    [
        [
            "Crop",
            "Month_Start",
            "Actual_Price",
            "Predicted_Price",
            "Absolute_Error",
            "Percentage_Error"
        ]
    ]
)


print("\n" + "=" * 70)
print("20 LARGEST PREDICTION ERRORS")
print("=" * 70)

display(
    largest_errors.round(2)
)


# ============================================================
# 23. SAVE TRAIN / TEST DATA
# ============================================================

train_df.to_csv(
    OUTPUT_DIR
    / "forecasting_train_data.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIR
    / "forecasting_test_data.csv",
    index=False
)


# ============================================================
# 24. SAVE BEST MODEL
# ============================================================

best_model_path = (
    MODEL_DIR
    / "food_price_forecasting_best_model.joblib"
)

joblib.dump(
    best_model,
    best_model_path
)


# ============================================================
# 25. SAVE FEATURE INFORMATION
# ============================================================

feature_info = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Type": [
            (
                "Categorical"
                if col in categorical_features
                else "Numeric"
            )
            for col in feature_columns
        ]
    }
)

feature_info.to_csv(
    OUTPUT_DIR
    / "model_feature_information.csv",
    index=False
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FORECASTING PIPELINE COMPLETE")
print("=" * 70)

print(
    f"\nTotal dataset rows:       {len(df):,}"
)

print(
    f"Training rows:            {len(train_df):,}"
)

print(
    f"Testing rows:             {len(test_df):,}"
)

print(
    f"Number of crops:          {df['Crop'].nunique()}"
)

print(
    f"Number of features:       {len(feature_columns)}"
)

print(
    f"Numeric features:         {len(numeric_features)}"
)

print(
    f"Categorical features:     {len(categorical_features)}"
)

print(
    f"\nBest ML model:            {best_model_name}"
)

print(
    f"Best model RMSE:          ₹{best_row['RMSE']:,.2f}"
)

print(
    f"Best model MAE:           ₹{best_row['MAE']:,.2f}"
)

print(
    f"Best model MAPE:          {best_row['MAPE']:.2f}%"
)

print(
    f"Best model SMAPE:         {best_row['SMAPE']:.2f}%"
)

print(
    f"Best model R²:            {best_row['R2']:.4f}"
)

print("\nSaved files:")

print(
    f"  ✓ {OUTPUT_DIR / 'model_comparison.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'crop_model_performance.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'price_band_model_performance.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'test_predictions.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'forecasting_train_data.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'forecasting_test_data.csv'}"
)

print(
    f"  ✓ {OUTPUT_DIR / 'model_feature_information.csv'}"
)

print(
    f"  ✓ {best_model_path}"
)

print("\n" + "=" * 70)
print("READY FOR MODEL EVALUATION")
print("=" * 70)